In [64]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [65]:
import numpy as np
import SI_CoRT
import CoRT_builder

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)  # For reproducibility


## Testing p_value

In [66]:
n_target = 50
n_source = 100
p = 200
K = 5
Ka = 2
h = 15
alpha = 0.05
T = 3
s_len = 10
s_vector = [0.5] * s_len

para_results_storage = []
CoRT_model = CoRT_builder.CoRT(0)
iteration = 1000

cnt1 = 0
cnt2 = 0
cnt3 = 0
cnt4 = 0

for i in range(iteration):
    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
    result_dict = SI_CoRT.SI_parametric(n_target, p, K, target_data, source_data, T, s_len)
    # print(f"P_value[{i}]: {result_dict['p_value']}")
    if result_dict != None:
        cnt1 += (result_dict['is_signal'] == True)
        cnt2 += (result_dict['is_signal'] == False)
        cnt3 += (result_dict['is_signal'] == True and result_dict['p_value'] <= alpha)
        cnt4 += (result_dict['is_signal'] == False and result_dict['p_value'] <= alpha)
        if i % 1 == 0:
            print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
            print(f"FPR: {cnt4 / cnt2}, TPR: {cnt3 / cnt1}")
            print(f"is_not_signal: {int(cnt2), int(cnt4)}")
            print(f"is_signal: {int(cnt1), int(cnt3)}")
            # print("\n")
            print("===========================================================================")
            
        para_results_storage.append(result_dict)

is_signal : True, p_values[0]: 0.9600028189747478
FPR: nan, TPR: 0.0
is_not_signal: (0, 0)
is_signal: (1, 0)
is_signal : True, p_values[1]: 0.054407704984216876
FPR: nan, TPR: 0.0
is_not_signal: (0, 0)
is_signal: (2, 0)
is_signal : True, p_values[2]: 0.0001331276432143813
FPR: nan, TPR: 0.3333333333333333
is_not_signal: (0, 0)
is_signal: (3, 1)


KeyboardInterrupt: 

In [ ]:
is_signal_cases = [r for r in para_results_storage if r['is_signal']]
not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
print(f"len not_signal_cases : {len(not_signal_cases)}")
print(f"false_positives: {false_positives}")
fpr = false_positives / len(not_signal_cases)
print(f"FPR: {fpr:.4f} (Target: {alpha})")
print("\n")
true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
print(f"len not_signal_cases : {len(is_signal_cases)}")
print(f"true_positives: {true_positives}")
tpr = true_positives / len(is_signal_cases)
print(f"TPR: {tpr:.4f}")

len not_signal_cases : 1
false_positives: 0
FPR: 0.0000 (Target: 0.05)


len not_signal_cases : 9
true_positives: 6
TPR: 0.6667
